In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from llm_judge import create_log_judge_agent, format_judge_prompt

In [3]:
import json

with open("evals_run_2026_03_06_results.json", "r") as f:
    results = json.load(f)

print(f"Loaded {len(results)} results.")

Loaded 27 results.


In [4]:
judge = create_log_judge_agent()

In [5]:
row = results[0]

In [6]:
prompt = format_judge_prompt(row)

In [ ]:
eval_result = await judge.run(prompt)
eval_output = eval_result.output

In [14]:
print(eval_output.label)
print(eval_output.reasoning)

good
1. The response provides a comprehensive step-by-step guide on how to set up an LLM as a judge, which accurately addresses the user's question. 
2. It incorporates relevant code snippets and explanations for each step, ensuring clarity and completeness. 
3. All tools were utilized appropriately: the search tool for gathering general information and the file retrieval tools to access specific examples that guided the provided instructions. 
4. The formatting is clean and follows good markdown practices, using headings, code blocks, and bullet points to enhance readability. 

Overall, the agent's response meets all the criteria for a 'good' classification.


In [7]:
async def run_eval(row):
    prompt = format_judge_prompt(row)
    eval_result = await judge.run(prompt)
    eval_output = eval_result.output
    return eval_output

In [8]:
from tqdm.auto import tqdm

In [9]:
eval_results = []

for row in tqdm(results):
    eval_output = await run_eval(row)
    eval_results.append({
        'row': row,
        'label': eval_output.label,
        'reasoning': eval_output.reasoning
    })

  0%|          | 0/27 [00:00<?, ?it/s]

In [10]:
rows = []

for result in eval_results:
    row = {
        'question': result['row']['input']['question'],
        'llm_answer': result['row']['rag_response']['answer'],
        'human_label': result['row']['label'],
        'human_comment': result['row']['comments'],
        'llm_label': result['label'],
        'llm_reasoning': result['reasoning']
    }
    rows.append(row)

In [11]:
import pandas as pd
df_rows = pd.DataFrame(rows)

In [12]:
df_rows.llm_label.value_counts()

llm_label
good    23
bad      4
Name: count, dtype: int64

In [13]:
df_rows.human_label.value_counts()

human_label
good    18
bad      9
Name: count, dtype: int64

In [14]:
(df_rows.llm_label == df_rows.human_label).mean()

np.float64(0.8148148148148148)

In [34]:
df_disagreement = df_rows[df_rows.llm_label != df_rows.human_label]

In [35]:
df_disagreement

,question,llm_answer,human_label,human_comment,llm_label,llm_reasoning
8,How do I save my results as a PDF?,"To save your results as a PDF in Evidently, yo...",bad,the pdf conversion is not based on the actual ...,good,The agent's response provides a comprehensive ...
9,How do I add a panel to my dashboard?,"To add a panel to your dashboard, you can do t...",bad,And so the agent didn't say that the example U...,good,The agent's response provides a detailed and a...
12,How do I set up alerts?,"To set up alerts, follow these steps:\n\n1. **...",bad,not clear that it's in evidently cloud - need ...,good,The agent's response accurately details the pr...
14,How do I create a new project?,"To create a new project, you can use the follo...",bad,overall good but have strange symbols: plus,good,The agent's response accurately and completely...
16,How do I integrate Evidently with MLflow?,"To integrate Evidently with MLflow, you can le...",bad,hallucinated reference of mlflow,good,The agent's response effectively addresses the...
17,How do I use Evidently with streaming data?,"To use Evidently with streaming data, you'll n...",bad,hallucinated it,good,The agent's response provides a thorough and c...
22,What is the Queen's Gambit in chess?,"This question is off-topic, and I only provide...",good,,bad,The agent's response is incorrect because it f...
23,What caused the Cuban Missile Crisis?,The question about the causes of the Cuban Mis...,good,good as it's off topic,bad,The agent's response is 'bad' because it compl...
24,How do I train a random forest model?,To train a Random Forest model using Evidently...,bad,the question is off-topic and it didn't answer it,good,The agent's response accurately answers the us...
25,How do I fix class imbalance in my dataset?,"To address class imbalance in your dataset, yo...",bad,hallucianated,good,The agent's response provides a range of techn...
